## Splitting & exploding arrays — semi-structured → rows

Bronze often lands array and string-list columns straight from JSON. Turning them into clean silver rows uses three primitives the exam tests directly.

- **`split(col, pattern)`** — turns a string into an array. `'travel,food,fuel'` → `['travel','food','fuel']`.
- **`explode(arr)`** — one output row per array element. The classic line-items / events-array pattern.
- **`posexplode(arr)`** — same as `explode`, but also emits the position index as `pos`.

```python
from pyspark.sql.functions import split, explode

# 'grocery,fuel' → ['grocery','fuel'] → two rows
df.withColumn("tags", split("merchant_tags", ",")) \
  .withColumn("tag", explode("tags")) \
  .drop("tags")
```

**The predicate version** — when you only need to *filter*, don't explode every row. `array_contains(arr, value)` tests membership in place:

```sql
WHERE array_contains(tags, 'travel')
```

**The inverse — collecting rows back into an array** inside a `groupBy`: `collect_list(col)` keeps duplicates (order undefined), and `collect_set(col)` deduplicates. Together they let you flatten out for processing and re-nest for a compact gold row.
